# 02. Stack Applications: Expressions, Decoding, Design

Stacks evaluate **nested** or **ordered** things: expressions, paths, nested strings.

**Learning order:** RPN → CALCULATOR → CALCULATOR WITH BRACKETS → DECODE STRING → SIMPLIFY PATH → MIN STACK → ASTEROIDS → INTERVIEW THINKING

## 1. Evaluate Reverse Polish Notation

`["2","1","+","3","*"]` means `(2 + 1) * 3`. Numbers are pushed. An operator pops the **two** top numbers, applies the operation, and pushes the result. Careful: the **second popped** is the left operand. Division truncates toward zero.

In [1]:
def eval_rpn(tokens):
    stack = []
    for t in tokens:
        if t in "+-*/" and len(t) == 1:
            b, a = stack.pop(), stack.pop()
            if t == "+": stack.append(a + b)
            elif t == "-": stack.append(a - b)
            elif t == "*": stack.append(a * b)
            else: stack.append(int(a / b))
        else:
            stack.append(int(t))
    return stack[0]

print(eval_rpn(["2", "1", "+", "3", "*"]))    # 9
print(eval_rpn(["4", "13", "5", "/", "+"]))   # 6

9
6


## 2. Basic Calculator II (`+ - * /`, no brackets)

Remember the **previous sign**. When you finish reading a number:
- `+` → push `num`; `-` → push `-num`
- `*` or `/` → pop the top, combine with `num`, push the result (higher precedence done immediately)

At the end, the answer is the **sum of the stack**.

In [2]:
def calculate_ii(s):
    stack, num, sign = [], 0, "+"
    for i, ch in enumerate(s):
        if ch.isdigit():
            num = num * 10 + int(ch)
        if (not ch.isdigit() and ch != " ") or i == len(s) - 1:
            if sign == "+":
                stack.append(num)
            elif sign == "-":
                stack.append(-num)
            elif sign == "*":
                stack.append(stack.pop() * num)
            else:
                stack.append(int(stack.pop() / num))
            sign, num = ch, 0
    return sum(stack)

print(calculate_ii("3+2*2"), calculate_ii(" 3/2 "), calculate_ii(" 3+5 / 2 "))   # 7 1 5

7 1 5


## 3. Basic Calculator with brackets (`+ - ( )`)

Keep a running `result` and the current `sign`. On `(`, **save** `result` and `sign` on the stack and start fresh. On `)`, finish the number, multiply by the saved sign, add the saved result.

In [3]:
def calculate(s):
    stack, res, num, sign = [], 0, 0, 1
    for ch in s:
        if ch.isdigit():
            num = num * 10 + int(ch)
        elif ch in "+-":
            res += sign * num
            num, sign = 0, 1 if ch == "+" else -1
        elif ch == "(":
            stack.append(res)
            stack.append(sign)
            res, sign = 0, 1
        elif ch == ")":
            res += sign * num
            num = 0
            res *= stack.pop()          # sign before the bracket
            res += stack.pop()          # result before the bracket
    return res + sign * num

print(calculate("1 + 1"), calculate(" 2-1 + 2 "), calculate("(1+(4+5+2)-3)+(6+8)"), calculate("-(2+3)"))   # 2 3 23 -5

2 3 23 -5


## 4. Decode String: `3[a2[c]]` → `accaccacc`

On `[`, push the **string built so far** and the **repeat count**, start a new string. On `]`, pop them and combine: `previous + current * count`.

In [4]:
def decode_string(s):
    stack, cur, num = [], "", 0
    for ch in s:
        if ch.isdigit():
            num = num * 10 + int(ch)
        elif ch == "[":
            stack.append((cur, num))
            cur, num = "", 0
        elif ch == "]":
            prev, count = stack.pop()
            cur = prev + cur * count
        else:
            cur += ch
    return cur

print(decode_string("3[a2[c]]"))        # accaccacc
print(decode_string("2[abc]3[cd]ef"))   # abcabccdcdcdef

accaccacc
abcabccdcdcdef


## 5. Simplify Path (Unix)

Split by `/`. Ignore empty parts and `.`. `..` pops the stack (go up one folder) if possible. Everything else is a folder name.

In [5]:
def simplify_path(path):
    stack = []
    for part in path.split("/"):
        if part == "..":
            if stack:
                stack.pop()
        elif part and part != ".":
            stack.append(part)
    return "/" + "/".join(stack)

print(simplify_path("/home//foo/"), simplify_path("/a/./b/../../c/"), simplify_path("/../"))   # /home/foo /c /

/home/foo /c /


## 6. Min Stack (design): `getMin()` in O(1)

Store with each value **the minimum so far** (`(value, current_min)`). Popping removes both, so the previous minimum is automatically restored.

In [6]:
class MinStack:
    def __init__(self):
        self.stack = []
    def push(self, val):
        cur_min = min(val, self.stack[-1][1]) if self.stack else val
        self.stack.append((val, cur_min))
    def pop(self):
        self.stack.pop()
    def top(self):
        return self.stack[-1][0]
    def get_min(self):
        return self.stack[-1][1]

m = MinStack()
m.push(-2); m.push(0); m.push(-3)
print(m.get_min())   # -3
m.pop()
print(m.top(), m.get_min())   # 0 -2

-3
0 -2


## 7. Asteroid Collision

Positive = moving right, negative = moving left. Only a **positive on the stack top vs a negative coming** collide. The bigger survives, equal destroys both.

In [7]:
def asteroid_collision(asteroids):
    stack = []
    for a in asteroids:
        alive = True
        while alive and a < 0 and stack and stack[-1] > 0:
            if stack[-1] < -a:
                stack.pop()                 # top explodes, keep checking
            elif stack[-1] == -a:
                stack.pop()
                alive = False               # both explode
            else:
                alive = False               # new one explodes
        if alive:
            stack.append(a)
    return stack

print(asteroid_collision([5, 10, -5]), asteroid_collision([8, -8]), asteroid_collision([10, 2, -5]))   # [5, 10] [] [10]

[5, 10] [] [10]


## 8. Interview thinking

**Signals:** expression evaluation, nested encoded strings, path simplification, "design a stack with extra operation in O(1)", collisions that cascade.

**Say:** "A stack keeps the context I need to resume after a nested part ends."

**Common mistakes:** operand order in `-` and `/` (second popped is the left side), integer division of negatives (`int(a / b)` truncates toward zero, `//` floors), not flushing the last number, forgetting the sign when a bracket opens.

## Quick Revision

- RPN: pop `b` then `a`, push `a op b`.
- Calculator: remember previous sign, `* /` combine immediately, answer = sum of the stack.
- Brackets: save `(result, sign)` on `(`, restore on `)`.
- Decode: save `(string, count)` on `[`.
- Min stack: store `(value, min so far)`.